In [ ]:
# 라이브러리 설치 (Faker: 가짜 데이터 생성을 도와줌)
!pip install -q Faker

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 23.4 MB/s eta 0:00:00


In [ ]:
# 파인튜닝 테스트용 가상의 데이터 1000개 샘플 생성
import pandas as pd
import random
import os
from faker import Faker

# Faker 한국어 설정
fake = Faker('ko_KR')

# 2. (주)휴먼퓨처소프트 (HumanFutureSoft) 기본 설정 (가상)
COMPANY_NAME = "휴먼퓨처소프트"
CEO_NAME = fake.name() # CEO 이름 (랜덤 생성)
FOUNDED_YEAR = 2019
LOCATION = fake.city() + " " + fake.street_address() # 본사 위치 (랜덤 생성)

PRODUCTS = [
    {
        "name": "AURA (Advanced Unified Reasoning Assistant)",
        "desc": "차세대 초거대 AI 모델을 기반으로 한 기업용 LLM 솔루션입니다. AGI에 근접한 추론 능력을 제공합니다.",
        "features": ["고급 추론", "멀티모달 처리", "AI 에이전트 생성"]
    },
    {
        "name": "TeamSync (팀싱크)",
        "desc": "프로젝트 관리를 위한 올인원 협업 툴입니다. 칸반 보드, 간트 차트, 실시간 문서를 통합 제공합니다.",
        "features": ["프로젝트 관리", "실시간 협업", "업무 자동화"]
    },
    {
        "name": "Future-Edu (퓨처에듀)",
        "desc": "AI 기반의 맞춤형 코딩 교육 플랫폼입니다. 개인의 학습 속도에 맞춰 커리큘럼을 동적으로 조절합니다.",
        "features": ["AI 튜터", "맞춤형 커리큘럼", "실시간 코드 리뷰"]
    }
]

BENEFITS = ["주 4일제(금요일 휴무) 시범 운영", "포괄임금제 폐지", "전직원 스톡옵션 지급", "근무지 자율 선택(국내/해외)", "업계 최고 수준 연봉", "개인 법인카드 지급(월 50만원)", "무제한 유급 휴가"]

# 3. 질문/답변 템플릿 정의
qa_templates = [
    # --- 회사 기본 정보 (약 25%) ---
    ("질문", "답변", 0.25),
    ("{company_name}는 어떤 회사인가요?", "{company_name}는 {location}에 위치한 AGI 및 LLM 솔루션 전문 기업으로, {founded_year}년에 {ceo_name} 대표가 설립했습니다."),
    ("{company_name}에 대해 알려주세요.", "저희 {company_name}는 '미래의 소프트웨어, 오늘을 살다'는 비전 아래, 가장 진보된 AI 기술을 연구하고 상용화하고 있습니다."),
    ("휴먼퓨처소프트의 비전이 무엇인가요?", "{company_name}의 비전은 인간과 AI가 완벽하게 공존하며 시너지를 내는 미래를 소프트웨어로 구현하는 것입니다."),
    ("휴먼퓨처소프트의 CEO는 누구인가요?", "현재 {company_name}의 대표이사(CEO)는 {ceo_name} 님입니다."),
    ("휴먼퓨처소프트는 언제 설립되었나요?", "{company_name}는 {founded_year}년에 설립되었습니다."),
    ("휴먼퓨처소프트 본사는 어디에 있나요?", "저희 본사는 {location}에 위치해 있습니다."),

    # --- 제품 및 서비스 (약 35%) ---
    ("질문", "답변", 0.35),
    ("{company_name}의 주요 제품은 무엇인가요?", "{company_name}는 {product_names} 등 시대를 앞서가는 AI 솔루션을 제공합니다."),
    ("{product_name}은 어떤 제품인가요?", "{product_desc}"),
    ("{product_name}의 주요 기능은 무엇인가요?", "{product_name}의 주요 기능으로는 {product_features} 등이 있습니다."),
    ("기업용 LLM 솔루션 이름이 뭔가요?", "저희 차세대 기업용 LLM 솔루션의 이름은 AURA(Advanced Unified Reasoning Assistant)입니다."),
    ("협업 툴도 만드나요?", "네, {company_name}는 올인원 협업 툴인 'TeamSync (팀싱크)'를 서비스하고 있습니다."),
    ("AI 교육 플랫폼이 있나요?", "네, AI 맞춤형 코딩 교육 플랫폼인 'Future-Edu (퓨처에듀)'도 저희 {company_name}의 핵심 서비스입니다."),
    ("{product_name} 가격은 얼마인가요?", "{product_name}의 가격은 플랜별로 다릅니다. 자세한 내용은 공식 홈페이지의 요금 안내 페이지를 참고해 주시거나, 영업팀에 문의해 주세요."),

    # --- 채용 및 복지 (약 30%) ---
    ("질문", "답변", 0.30),
    ("{company_name} 채용 절차는 어떻게 되나요?", "{company_name}의 채용 절차는 '서류 전형 > 온라인 코딩 테스트 > 1차 기술 인터뷰 > 2차 컬처핏 인터뷰 > 최종 합격' 순으로 진행됩니다."),
    ("{company_name} 인재상이 궁금합니다.", "저희는 '미래 지향성', '완전한 자율성', '빠른 실행력'을 핵심 가치로 삼으며, 이러한 인재상을 가진 분을 찾고 있습니다."),
    ("휴먼퓨처소프트 복지는 어떤가요?", "휴먼퓨처소프트는 {benefit_list} 등 임직원의 삶과 성장을 위한 파격적인 복지를 제공하고 있습니다."),
    ("휴먼퓨처소프트는 주 4일제 하나요?", "네, 저희는 {benefit_list_snippet} 등 선진적인 근무 환경을 도입 중이며, 현재 주 4일제를 시범 운영하고 있습니다."),
    ("휴먼퓨처소프트 개발자 채용 중인가요?", "네, {company_name}는 AGI 연구원, LLM 엔지니어 등 다양한 직군의 인재를 상시 채용 중입니다. 공식 채용 사이트를 확인해 주세요."),
    ("{company_name}의 근무 환경은 어떤가요?", "{company_name}는 완전한 자율과 책임을 중시하며, {benefit_list_snippet} 등을 통해 최고의 몰입 환경을 제공합니다."),

    # --- 기타/고객지원 (약 10%) ---
    ("질문", "답변", 0.10),
    ("{product_name} 사용 중 문제가 생겼습니다. 어디로 연락해야 하나요?", "{product_name} 관련 기술 지원은 저희 고객센터(support@humanfuture.ai) 또는 1588-1000으로 연락 주시면 신속히 도와드리겠습니다."),
    ("휴먼퓨처소프트의 최근 소식이 궁금합니다.", "휴먼퓨처소프트의 최신 뉴스와 업데이트는 공식 블로그나 뉴스룸을 참고해 주세요."),
    ("휴먼퓨처소프트의 파트너가 되고 싶습니다.", "파트너십 문의는 공식 홈페이지의 '제휴 문의' 채널을 통해 접수해 주시면 담당자가 검토 후 연락드리겠습니다.")
]

def generate_data(num_rows=1000):
    data = []

    # 가중치 기반 카테고리 선택 준비
    categories = []
    weights = []
    current_category_templates = []

    for item in qa_templates:
        if item[0] == "질문":
            if current_category_templates:
                categories.append(current_category_templates)
                weights.append(last_weight)
            current_category_templates = []
            last_weight = item[2]
        else:
            current_category_templates.append(item)
    categories.append(current_category_templates)
    weights.append(last_weight)

    for _ in range(num_rows):
        category = random.choices(categories, weights=weights, k=1)[0]
        q_template, a_template = random.choice(category)

        product = random.choice(PRODUCTS)
        product_names_list = [p['name'] for p in PRODUCTS]
        random.shuffle(product_names_list)
        product_names_str = ", ".join(product_names_list)

        benefit_sample = random.sample(BENEFITS, k=random.randint(2, 3))
        benefit_list_str = ", ".join(benefit_sample)

        replacements = {
            "company_name": COMPANY_NAME,
            "ceo_name": CEO_NAME,
            "founded_year": str(FOUNDED_YEAR),
            "location": LOCATION,
            "product_name": product['name'],
            "product_desc": product['desc'],
            "product_features": ", ".join(product['features']),
            "product_names": product_names_str,
            "benefit_list": ", ".join(BENEFITS),
            "benefit_list_snippet": benefit_list_str,
        }

        q = q_template.format(**replacements)
        a = a_template.format(**replacements)

        data.append({"질문": q, "답변": a})

    return pd.DataFrame(data)

# --- 실행 ---
NUM_ROWS_TO_GENERATE = 1000 # 샘플수
OUTPUT_DIR = "dataset"
OUTPUT_FILENAME = os.path.join(OUTPUT_DIR, "indata_kor.csv")

os.makedirs(OUTPUT_DIR, exist_ok=True)
df = generate_data(NUM_ROWS_TO_GENERATE)

# 한글 깨짐 방지 "utf-8-sig"로 저장
df.to_csv(OUTPUT_FILENAME, index=False, encoding='utf-8-sig')

print(f"가상 데이터 {len(df)}개 생성 완료!")
print(f"파일이 '{OUTPUT_FILENAME}'에 저장되었습니다. (Excel 호환 `utf-8-sig` 적용)")

# --- 생성된 데이터 샘플 10개 확인 ---
print("\n--- [생성된 데이터 샘플 (상위 10개)] ---")
print(df.head(10))

가상 데이터 1000개 생성 완료!
파일이 'dataset/indata_kor.csv'에 저장되었습니다. (Excel 호환 `utf-8-sig` 적용)

--- [생성된 데이터 샘플 (상위 10개)] ---
                       질문                                                 답변
0    휴먼퓨처소프트의 CEO는 누구인가요?                   현재 휴먼퓨처소프트의 대표이사(CEO)는 이지원 님입니다.
1     휴먼퓨처소프트는 주 4일제 하나요?  네, 저희는 개인 법인카드 지급(월 50만원), 포괄임금제 폐지 등 선진적인 근무 ...
2             협업 툴도 만드나요?  네, 휴먼퓨처소프트는 올인원 협업 툴인 'TeamSync (팀싱크)'를 서비스하고 ...
3    휴먼퓨처소프트 본사는 어디에 있나요?   저희 본사는 정선군 인천광역시 중구 학동거리 지하150 (서준박리)에 위치해 있습니다.
4     휴먼퓨처소프트 인재상이 궁금합니다.  저희는 '미래 지향성', '완전한 자율성', '빠른 실행력'을 핵심 가치로 삼으며,...
5  휴먼퓨처소프트의 주요 제품은 무엇인가요?  휴먼퓨처소프트는 Future-Edu (퓨처에듀), TeamSync (팀싱크), AU...
6       휴먼퓨처소프트 복지는 어떤가요?  휴먼퓨처소프트는 주 4일제(금요일 휴무) 시범 운영, 포괄임금제 폐지, 전직원 스톡...
7    기업용 LLM 솔루션 이름이 뭔가요?  저희 차세대 기업용 LLM 솔루션의 이름은 AURA(Advanced Unified ...
8     휴먼퓨처소프트의 비전이 무엇인가요?  휴먼퓨처소프트의 비전은 인간과 AI가 완벽하게 공존하며 시너지를 내는 미래를 소프트...
9      휴먼퓨처소프트는 어떤 회사인가요?  휴먼퓨처소프트는 정선군 인천광역시 중구 학동거리 지하150 (서준박리)에 위치한 A...
